In [52]:
import numpy as np
import pyslha
import glob
from collections import namedtuple


### Get Widths

In [56]:
pars = namedtuple('parameters',['M1','M0','lambdaUV','w1'])
parameters = []
for f in glob.glob('./width_results/*.dat'):
    slhaData = pyslha.readSLHAFile(f)
    p = pars(M1=slhaData.blocks['MASS'][4000023], M0=slhaData.blocks['MASS'][4000022], lambdaUV=slhaData.blocks['NPINPUTS'][1], w1=slhaData.decays[4000023].totalwidth)
    parameters.append(p)

In [57]:
mSs = [400, 450, 600, 800, 1000] #mass of scalar particle [GeV]
beta1s = [0.1, 0.2, 0.25, 0.3, 0.4] #boost of the chi1 in the labframe
Dm21s = [5] #mass difference between chi2 and chi1 [GeV]
Dm10s = [60, 100] #mass difference between chi1 and chi0 [GeV] (come back and try 100)
ctau0s = [20] #proper lifetime of Chi1 [m]

In [60]:
pts = []
lambdaUV0 = 9.07e6
for mS in mSs:
    for beta1 in beta1s:
        for Dm21 in Dm21s:
            m1_solutions = [ (-beta1 ** 2 * Dm21 ** 3 + sgn * (-beta1 ** 2 * Dm21 ** 4 * mS ** 2 + 2 * beta1 ** 2 * Dm21 ** 2 * mS ** 4 - beta1 ** 2 * mS ** 6 + Dm21 ** 4 * mS ** 2 - 2 * Dm21 ** 2 * mS ** 4 + mS ** 6) ** .5 + beta1 ** 2 * Dm21 * mS ** 2 + Dm21 ** 3 - Dm21 * mS ** 2)/(2 * (beta1 ** 2 * Dm21 ** 2 - Dm21 ** 2 + mS ** 2)) for sgn in [+1 , -1]]
            M1 = max(m1_solutions)
            M2 = M1 + Dm21 #mass of chi2 [GeV]
            for Dm10 in Dm10s:
                M0 = M1 - Dm10 #mass of chi0 [GeV]
                if sorted([mS,M2,M1,M0],reverse=True) != [mS,M2,M1,M0]:
                    print(f"Skipping point: mS={mS}, M2={M2}, M1={M1}, M0={M0} because it does not satisfy mS > M2 > M1 > M0")
                    continue
                p = [par for par in parameters if np.isclose(par.M1, M1,rtol=1e-3) and np.isclose(par.M0, M0,rtol=1e-3)]
                if not p:
                    print(f"Skipping point: mS={mS}, M2={M2}, M1={M1}, M0={M0} because it does not have a corresponding parameter file")
                    continue
                p = p[0]
                w10 = p.w1
                lambdaUV0 = p.lambdaUV
                for ctau0_m in ctau0s:
                    w1 = 6.58*3.e-17/ctau0_m
                    lambdaUV = lambdaUV0/np.sqrt(w1/w10)
                    pt = [float(f"{x:1.4g}") if x < 1e4 else float(f"{x:1.4e}") for x in (mS, M2, M1, M0, lambdaUV, ctau0_m)]
                    pts.append(tuple(pt))
                    

In [38]:
print(len(pts))
print(pts)

50
[(400.0, 201.5, 196.5, 136.5, 9070000.0, 20.0), (400.0, 201.5, 196.5, 96.52, 9070000.0, 20.0), (400.0, 198.6, 193.6, 133.6, 9070000.0, 20.0), (400.0, 198.6, 193.6, 93.56, 9070000.0, 20.0), (400.0, 196.3, 191.3, 131.3, 9070000.0, 20.0), (400.0, 196.3, 191.3, 91.3, 9070000.0, 20.0), (400.0, 193.5, 188.5, 128.5, 9070000.0, 20.0), (400.0, 193.5, 188.5, 88.51, 9070000.0, 20.0), (400.0, 186.2, 181.2, 121.2, 9070000.0, 20.0), (400.0, 186.2, 181.2, 81.2, 9070000.0, 20.0), (450.0, 226.4, 221.4, 161.4, 9070000.0, 20.0), (450.0, 226.4, 221.4, 121.4, 9070000.0, 20.0), (450.0, 223.1, 218.1, 158.1, 9070000.0, 20.0), (450.0, 223.1, 218.1, 118.1, 9070000.0, 20.0), (450.0, 220.5, 215.5, 155.5, 9070000.0, 20.0), (450.0, 220.5, 215.5, 115.5, 9070000.0, 20.0), (450.0, 217.4, 212.4, 152.4, 9070000.0, 20.0), (450.0, 217.4, 212.4, 112.4, 9070000.0, 20.0), (450.0, 209.1, 204.1, 144.1, 9070000.0, 20.0), (450.0, 209.1, 204.1, 104.1, 9070000.0, 20.0), (600.0, 301.0, 296.0, 236.0, 9070000.0, 20.0), (600.0, 301

In [47]:
pts_m1 = []
for pt in pts:
    p = (pt[2],pt[3], pt[4])
    if p not in pts_m1:
        pts_m1.append(p)

In [49]:
print(len(pts_m1))
print(pts_m1)

50
[(196.5, 136.5, 9070000.0), (196.5, 96.52, 9070000.0), (193.6, 133.6, 9070000.0), (193.6, 93.56, 9070000.0), (191.3, 131.3, 9070000.0), (191.3, 91.3, 9070000.0), (188.5, 128.5, 9070000.0), (188.5, 88.51, 9070000.0), (181.2, 121.2, 9070000.0), (181.2, 81.2, 9070000.0), (221.4, 161.4, 9070000.0), (221.4, 121.4, 9070000.0), (218.1, 158.1, 9070000.0), (218.1, 118.1, 9070000.0), (215.5, 155.5, 9070000.0), (215.5, 115.5, 9070000.0), (212.4, 152.4, 9070000.0), (212.4, 112.4, 9070000.0), (204.1, 144.1, 9070000.0), (204.1, 104.1, 9070000.0), (296.0, 236.0, 9070000.0), (296.0, 196.0, 9070000.0), (291.5, 231.5, 9070000.0), (291.5, 191.5, 9070000.0), (288.1, 228.1, 9070000.0), (288.1, 188.1, 9070000.0), (283.9, 223.9, 9070000.0), (283.9, 183.9, 9070000.0), (272.9, 212.9, 9070000.0), (272.9, 172.9, 9070000.0), (395.5, 335.5, 9070000.0), (395.5, 295.5, 9070000.0), (389.5, 329.5, 9070000.0), (389.5, 289.5, 9070000.0), (385.0, 325.0, 9070000.0), (385.0, 285.0, 9070000.0), (379.3, 319.3, 9070000.0),